In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor

In [2]:
df = pd.read_csv("TV.csv")
df.shape

(666, 11)

In [3]:
df.head(1)

,Product_Name,Stars,Ratings,Reviews,current_price,MRP,channel,Operating_system,Picture_quality,Speaker,Frequency
0,Croma,4.2,1773,217,7990,20000,HD Ready 1366 x 768 Pixels,20 Speaker Output,60 Hz Refresh Rate,2 x HDMI | 2 x USB,1 Year Warranty


### Q1.

In [ ]:
df_q1 = df[["Frequency", "Picture_quality", "Speaker"]].copy()
df_q1.head(1)

In [ ]:
# ser_u = df_q1["Frequency"].drop_duplicates()
# ser_u = df_q1["Picture_quality"].drop_duplicates()
ser_u = df_q1["Speaker"].drop_duplicates()
# ser_u[ser_u.str.contains("Hz")]
ser_u[ser_u.str.contains("[0-9]{2,3} Hz")] # 숫자 2~3자리, 한 칸 띄우고 "Hz"

In [11]:
df_q1["hz_ck1"] = df_q1["Frequency"].str.contains("Hz") + 0
df_q1["hz_ck2"] = df_q1["Picture_quality"].str.contains("Hz") + 0
df_q1["hz_ck3"] = df_q1["Speaker"].str.contains("Hz") + 0
df_q1["hz_ck_cnt"] = df_q1["hz_ck1"] + df_q1["hz_ck2"] + df_q1["hz_ck3"]

In [ ]:
df_q1.head()

In [ ]:
ser_test = pd.Series([True, False, True])
ser_test + 0

In [ ]:
df_q1["hz_ck_cnt"].value_counts()

In [16]:
df_q1_sub = df_q1.loc[df_q1["hz_ck_cnt"] != 0, ]

In [17]:
len(df_q1_sub)

662

In [ ]:
df_q1_sub.iloc[:2, :3]

In [20]:
ser_cnt = df_q1_sub.iloc[:, :3].apply(lambda x: x.str.contains("60 Hz").sum(), axis = 1)
ser_cnt.value_counts() # 510!!!

1    510
0    152
Name: count, dtype: int64

In [ ]:
df_hz = df_q1["Frequency"].str.extract("([0-9]{2,3} Hz)")
df_hz = df_hz.fillna("")
df_hz[0].str.len()

### Q2.

In [ ]:
df_q2 = df[["Stars", "Operating_system", "channel", "Picture_quality"]].copy()
df_q2.head(2)

In [ ]:
# ser_u = df_q2["Operating_system"].drop_duplicates()
# ser_u = df_q2["channel"].drop_duplicates()
ser_u = df_q2["Picture_quality"].drop_duplicates()
ser_u[ser_u.str.contains("4K|8K")]

In [35]:
df_q2["cnt_4k"] = df_q2.iloc[:, 1:4].apply(lambda x: x.str.contains("4K").sum(), axis = 1)
df_q2["cnt_8k"] = df_q2.iloc[:, 1:4].apply(lambda x: x.str.contains("8K").sum(), axis = 1)
df_q2.tail(2)

,Stars,Operating_system,channel,Picture_quality,cnt_4k,cnt_8k
664,0.0,Full HD 1920 x 1080 Pixels,Netflix|Prime Video|Apple TV|Disney+Hotstar|Yo...,16 Speaker Output,0,0
665,4.4,10W + 10W Speaker Output,Ultra HD (4K) 3840 x 2160 pixels Pixels,50 Hz Refresh Rate,1,0


In [36]:
df_q2["cnt_4k"].sum(), df_q2["cnt_8k"].sum()

(365, 1)

In [37]:
stat_4k = df_q2.loc[df_q2["cnt_4k"] != 0, "Stars"].mean()
stat_8k = df_q2.loc[df_q2["cnt_8k"] != 0, "Stars"].mean()
stat_4k, stat_8k

(3.2180821917808218, 3.6)

In [38]:
round(abs(stat_4k - stat_8k), 2)

0.38

### Q3.

In [40]:
df_q3 = df.loc[~df["channel"].str.contains("Pixel|Oper"), ].reset_index(drop = True)
df_q3["x1"] = df_q3["Reviews"] / df_q3["Ratings"] 
df_q3["x2"] = df_q3["MRP"]
df_q3["x3"] = df_q3["current_price"] / df_q3["MRP"]
df_q3["x4"] = df_q3["channel"].str.contains("Netflix") + 0
df_q3["x5"] = df_q3["channel"].str.contains("Prime Video") + 0
df_q3["x6"] = df_q3["Picture_quality"].str.contains("4K|8K") + 0

In [ ]:
df_q3.head(2)

In [42]:
df_model = df_q3[["Stars", "x1", "x2", "x3", "x4", "x5", "x6"]]
df_model.head(1)

,Stars,x1,x2,x3,x4,x5,x6
0,3.8,0.137941,21999,0.395427,1,0,0


In [44]:
df_model = df_model.dropna()

In [ ]:
df_model.isna().sum()

In [46]:
len(df_model)

197

In [ ]:
model_rf = RandomForestRegressor(random_state = 123) # 모델 정의
model_rf.fit(X = df_model.drop(columns = "Stars"), # X는 대문자!!
             y = df_model["Stars"])

In [49]:
model_rf.feature_importances_ # MDI

array([0.31220617, 0.17383963, 0.4446912 , 0.02714087, 0.02400442,
       0.01811773])

In [50]:
pd.Series(model_rf.feature_importances_, index = df_model.columns[1:])
# X3: 할인율

x1    0.312206
x2    0.173840
x3    0.444691
x4    0.027141
x5    0.024004
x6    0.018118
dtype: float64